# Inspect Augmentations

Visualise the custom augmentation pipeline on a single prepared tile.

**Usage:** set `TILE_PATH` to any `.npy` tile from your prepared dataset, then run all cells.

Three sections:
1. **Original tile** — image and all five masks.
2. **N augmented samples** — the full training pipeline applied N times, showing variability and confirming spatial consistency between image and masks.
3. **Each transform in isolation** — apply individual transforms to the same tile for debugging.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from aimsegdl.transforms.custom_transforms import (
    Compose, OneOf, Rotate, CenterCrop, RandomCrop,
    HorizontalFlip, VerticalFlip, RandomScale,
    GaussianBlur, MedianBlur, GaussNoise, Defocus,
    ColorJitter, RandomGamma, ToTensor, transforms_fn,
)
from aimsegdl.utils.image_processing import normalize

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
TILE_PATH = "../prepared_data/train_tiles/your_tile.npy"   # <-- set this
N_AUGMENTED_SAMPLES = 8
# ───────────────────────────────────────────────────────────────────────────

assert os.path.exists(TILE_PATH), f"Tile not found: {TILE_PATH}"
tile = np.load(TILE_PATH, allow_pickle=True).item()

image    = normalize(tile["image"]).astype(np.float32)
masks    = [
    tile["mask_fibre"],
    tile["mask_axon"],
    tile["mask_sem"],
    tile["sdt_fibre"],
    tile["sdt_axon"],
]
MASK_LABELS = ["mask_fibre", "mask_axon", "mask_sem", "sdt_fibre", "sdt_axon"]

img_h, img_w = image.shape[:2]
print(f"Tile shape: {image.shape}  |  dtype: {image.dtype}")
for name, m in zip(MASK_LABELS, masks):
    print(f"  {name}: shape={m.shape}  dtype={m.dtype}  min={m.min():.3f}  max={m.max():.3f}")

In [ ]:
# ── Plotting helpers ────────────────────────────────────────────────────────

def sdt_cmap():
    """Black for background (<=0), inferno gradient for foreground (>0)."""
    import matplotlib.cm as cm
    inferno = cm.get_cmap("inferno", 256)
    colors = inferno(np.linspace(0, 1, 256))
    colors[0] = [0, 0, 0, 1]
    return mcolors.LinearSegmentedColormap.from_list("sdt", colors, N=256)

SDT_CMAP = sdt_cmap()

def _show_img(ax, arr, title, cmap="gray"):
    if isinstance(arr, np.ndarray) and arr.ndim == 3:
        # CxHxW tensor converted to numpy: take first channel
        arr = arr[0]
    ax.imshow(arr, cmap=cmap, interpolation="nearest")
    ax.set_title(title, fontsize=8)
    ax.axis("off")

def _sdt_display(arr):
    """Clip to [0,1] range for SDT colourmap display."""
    clipped = np.clip(arr, 0, None)
    vmax = clipped.max()
    return clipped / (vmax + 1e-8) if vmax > 0 else clipped

def _cmaps_for(masks):
    return ["viridis", "viridis", "viridis", SDT_CMAP, SDT_CMAP]

def plot_row(ax_row, img, masks, row_title=None):
    """Plot one image + 5 masks as a row of 6 subplots."""
    if row_title:
        ax_row[0].set_ylabel(row_title, fontsize=7, rotation=0, labelpad=60, va="center")
    if hasattr(img, 'numpy'):
        img_np = img.numpy()
        img_display = img_np[0] if img_np.ndim == 3 else img_np
    else:
        img_display = img
    _show_img(ax_row[0], img_display, "image")
    cmaps = _cmaps_for(masks)
    for j, (m, name, cmap) in enumerate(zip(masks, MASK_LABELS, cmaps)):
        if hasattr(m, 'numpy'):
            m = m.numpy()
        disp = _sdt_display(m) if "sdt" in name else m
        _show_img(ax_row[j + 1], disp, name, cmap=cmap)

## 1 — Original tile

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
plot_row(axes, image, masks, row_title="original")
plt.suptitle("Original tile", fontsize=10)
plt.tight_layout()
plt.show()

## 2 — N augmented samples (full training pipeline)

Each row is one independent draw from the training transform.
Check that image and masks stay spatially aligned and that SDT fill values are `-1` (black) in rotated/scaled border regions.

In [ ]:
train_tf, _ = transforms_fn(img_h, img_w)

fig, axes = plt.subplots(N_AUGMENTED_SAMPLES, 6, figsize=(18, 3 * N_AUGMENTED_SAMPLES))
for i in range(N_AUGMENTED_SAMPLES):
    result = train_tf(image=image, masks=masks)
    plot_row(axes[i], result["image"], result["masks"], row_title=f"sample {i+1}")

plt.suptitle(f"Training augmentation — {N_AUGMENTED_SAMPLES} samples", fontsize=10)
plt.tight_layout()
plt.show()

## 3 — Each transform in isolation

Each row shows the original alongside the result of one transform applied with `p=1`.
Spatial transforms also show all masks; intensity-only transforms show the image only.

In [ ]:
_FILL = [0, 0, 0, -1, -1]

def apply_forced(transform, img, msks):
    """Apply a transform with p forced to 1.0."""
    orig_p = getattr(transform, 'p', 1.0)
    transform.p = 1.0
    result = transform(img, msks)
    transform.p = orig_p
    if isinstance(result, tuple):
        return result
    return result["image"], result["masks"]

spatial_transforms = [
    ("RandomScale (0.9–1.1)",  RandomScale(scale_range=(0.9, 1.1), p=1.0, fill_values=_FILL)),
    ("Rotate (±30°)",          Rotate(limit=30, p=1.0, fill_values=_FILL)),
    ("HorizontalFlip",         HorizontalFlip(p=1.0)),
    ("VerticalFlip",           VerticalFlip(p=1.0)),
]

intensity_transforms = [
    ("GaussianBlur",  GaussianBlur(sigma_range=(0.5, 2.0), p=1.0)),
    ("MedianBlur",    MedianBlur(size=3, p=1.0)),
    ("GaussNoise",    GaussNoise(var_limit=(0.05, 0.1), p=1.0)),
    ("Defocus",       Defocus(sigma_range=(1.0, 3.0), p=1.0)),
    ("ColorJitter",   ColorJitter(brightness=0.3, contrast=0.3, p=1.0)),
    ("RandomGamma",   RandomGamma(gamma_range=(0.8, 1.2), p=1.0)),
]

# Spatial: show original + augmented (image + all masks)
print("Spatial transforms")
for name, tf in spatial_transforms:
    aug_img, aug_masks = apply_forced(tf, image.copy(), [m.copy() for m in masks])

    # Crop to original size if upscaled (mirrors what RandomCrop does)
    aug_img_np = aug_img.numpy()[0] if hasattr(aug_img, 'numpy') else aug_img
    h_aug = aug_img_np.shape[0]
    w_aug = aug_img_np.shape[1]
    crop_h = min(h_aug, img_h)
    crop_w = min(w_aug, img_w)
    aug_img_np = aug_img_np[:crop_h, :crop_w]
    aug_masks_np = []
    for m in aug_masks:
        mn = m.numpy() if hasattr(m, 'numpy') else m
        aug_masks_np.append(mn[:crop_h, :crop_w])

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    plot_row(axes[0], image, masks, row_title="original")
    plot_row(axes[1], aug_img_np, aug_masks_np, row_title=name)
    plt.suptitle(name, fontsize=10)
    plt.tight_layout()
    plt.show()

# Intensity: show original image vs augmented image only (2 subplots)
print("Intensity transforms")
n_intensity = len(intensity_transforms)
fig, axes = plt.subplots(n_intensity, 2, figsize=(6, 3 * n_intensity))
for i, (name, tf) in enumerate(intensity_transforms):
    aug_img, _ = apply_forced(tf, image.copy(), masks)
    aug_img_np = aug_img.numpy()[0] if hasattr(aug_img, 'numpy') else aug_img
    _show_img(axes[i, 0], image, "original")
    _show_img(axes[i, 1], aug_img_np, name)
    axes[i, 0].set_ylabel(name, fontsize=7, rotation=0, labelpad=60, va="center")
plt.suptitle("Intensity transforms", fontsize=10)
plt.tight_layout()
plt.show()